# Pueblo, CO — LVT Policy Analysis

Models a revenue-neutral split-rate LVT shift for parcels inside the **City of Pueblo** corporate limits (Pueblo County, CO).

## Policy decisions (carried from PR #8)

| Choice | Setting | Note |
|---|---|---|
| Geographic scope | City of Pueblo (spatial filter against City_Limits boundary) | Source endpoint has no city attribute; `OwnerCity` is mailing address only |
| Reform shape | Split-rate at 2:1 (land:improvement) | Lars's choice |
| Land/improvement values | `LandAssessedValue` / `ImprovementsAssessedValue` | Pueblo County source schema |
| Exemption flag | `TaxExempt == 'yes'` | Lars's choice |
| Classification | `Zoning` field (mostly empty in source) | Fallback to single "Other" bucket |
| Current tax | **Proxy** uniform mill rate | Source `Tax` and `LevyURL` fields are blank for most parcels; Lars's URL-regex approach yielded NaN |

> **About the proxy mill rate.** Pueblo County's source endpoint returns blank
> `Tax` and `LevyURL` fields. Lars's PR #8 attempted to parse millage from
> `LevyURL` via regex but acknowledged most parcels would be null. Carrying
> forward a uniform 1.0-mill placeholder so the revenue-neutral solver produces
> meaningful relative shifts. Real Pueblo mill levies vary by `TaxDistrict`
> (e.g., 60B, 70L) and need to come from Pueblo County Auditor's published
> consolidated levy table. Absolute dollar values are not real.

> **Differences from Lars's draft.** Lars's notebook had `COL_CITY_NAME = None`
> (no city filter — would have modeled all 101K Pueblo County parcels). Added a
> spatial filter against the City_Limits boundary so the output is actually
> Pueblo *City*, not Pueblo *County*.


## Section 1 — Imports and setup

In [ ]:
import sys
import os
import time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from shapely.geometry import Polygon

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'pueblo'
STATE_FIPS = '08'
COUNTY_FIPS = '101'  # Pueblo County
MODEL_TYPE = 'split_rate_2to1'
LAND_IMPROVEMENT_RATIO = 2.0

DATA_DIR = Path('data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

PARCELS_ENDPOINT = (
    'https://services1.arcgis.com/IL17xsvNU5Bmw3RY/arcgis/rest/services/'
    'County_Parcels/FeatureServer/0/query'
)
# Pueblo's own ArcGIS "City_Limits_*" services either return null geometry or
# a 250m fragment. Use Census TIGER 2025 Incorporated Places instead — Pueblo
# city has Census place GEOID 0862000.
CITY_LIMITS_ENDPOINT = (
    'https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/'
    'Places_CouSub_ConCity_SubMCD/MapServer/4/query'
)
CITY_LIMITS_WHERE = "GEOID='0862000'"  # Pueblo, CO incorporated place
PAGE_SIZE = 2000

OUT_FIELDS = [
    'OBJECTID', 'PAR_TXT', 'PAR_NUM', 'OwnerCity', 'TaxDistrict', 'TaxExempt',
    'SeniorExemption', 'Zoning',
    'LandAssessedValue', 'LandActualValue',
    'ImprovementsAssessedValue', 'ImprovementsActualValue',
    'LevyURL',
]

## Section 2 — Fetch and spatially filter parcels

1. Fetch Pueblo City boundary polygon
2. Page through Pueblo County parcels (with geometry)
3. Spatial filter parcels whose representative point falls inside the city


In [ ]:
def fetch_arcgis_records(query_url, where, out_fields, page_size=2000, return_geometry=False):
    session = requests.Session()
    count_resp = session.get(
        query_url,
        params={'f': 'json', 'where': where, 'returnCountOnly': 'true'},
        timeout=60,
    )
    count_resp.raise_for_status()
    total = int(count_resp.json().get('count', 0))
    print(f'Total matching: {total:,}')
    rows = []
    for offset in range(0, total, page_size):
        params = {
            'f': 'json', 'where': where, 'outFields': ','.join(out_fields),
            'returnGeometry': str(return_geometry).lower(),
            'resultOffset': offset, 'resultRecordCount': page_size,
            'orderByFields': 'OBJECTID ASC',
        }
        if return_geometry:
            params['outSR'] = 4326
            params['geometryPrecision'] = 6
        resp = session.get(query_url, params=params, timeout=180)
        resp.raise_for_status()
        features = resp.json().get('features', [])
        if not features:
            break
        rows.extend(features)
    return rows


def esri_polygon_to_shapely(geom):
    rings = (geom or {}).get('rings')
    if not rings:
        return None
    return Polygon(rings[0], holes=rings[1:] if len(rings) > 1 else None)


def load_city_limits():
    cache = DATA_DIR / 'pueblo_city_limits.parquet'
    if cache.exists():
        return gpd.read_parquet(cache)
    features = fetch_arcgis_records(
        CITY_LIMITS_ENDPOINT, CITY_LIMITS_WHERE, ['GEOID', 'NAME'],
        page_size=PAGE_SIZE, return_geometry=True,
    )
    polys = []
    for f in features:
        g = esri_polygon_to_shapely(f.get('geometry') or {})
        if g is not None:
            polys.append(g)
    gdf = gpd.GeoDataFrame({'OBJECTID': range(len(polys))}, geometry=polys, crs='EPSG:4326')
    gdf.to_parquet(cache, index=False)
    return gdf


def load_county_parcels():
    cache = DATA_DIR / 'pueblo_county_parcels.parquet'
    if cache.exists():
        return gpd.read_parquet(cache)
    print('Fetching Pueblo County parcels (this will take several minutes — ~101K records)')
    features = fetch_arcgis_records(
        PARCELS_ENDPOINT, '1=1', OUT_FIELDS,
        page_size=PAGE_SIZE, return_geometry=True,
    )
    rows = []
    for f in features:
        attrs = f.get('attributes', {})
        g = esri_polygon_to_shapely(f.get('geometry') or {})
        if g is not None:
            attrs['geometry'] = g
            rows.append(attrs)
    gdf = gpd.GeoDataFrame(rows, geometry='geometry', crs='EPSG:4326')
    gdf.to_parquet(cache, index=False)
    return gdf


t0 = time.time()
city_limits = load_city_limits()
print(f'City limits polygons: {len(city_limits)} (loaded in {time.time()-t0:.1f}s)')

t0 = time.time()
county_parcels = load_county_parcels()
print(f'County parcels: {len(county_parcels):,} (loaded in {time.time()-t0:.1f}s)')

In [ ]:
# Spatial filter: keep parcels whose representative point is within city limits.
city_3857 = city_limits.to_crs(epsg=3857)[['geometry']]  # drop OBJECTID to avoid sjoin collision
parcels_3857 = county_parcels.to_crs(epsg=3857)
parcels_3857['rep_point'] = parcels_3857.geometry.representative_point()
points = gpd.GeoDataFrame(
    parcels_3857[['OBJECTID']].copy(),
    geometry=parcels_3857['rep_point'],
    crs=parcels_3857.crs,
)
joined = gpd.sjoin(points, city_3857, how='inner', predicate='within')
in_city_oids = set(joined['OBJECTID'].astype(int).tolist())

gdf = county_parcels[county_parcels['OBJECTID'].isin(in_city_oids)].copy().reset_index(drop=True)
print(f'Parcels inside Pueblo city limits: {len(gdf):,} / {len(county_parcels):,}')

numeric_cols = ['LandAssessedValue', 'LandActualValue',
                'ImprovementsAssessedValue', 'ImprovementsActualValue']
for col in numeric_cols:
    gdf[col] = pd.to_numeric(gdf[col], errors='coerce').fillna(0)

## Section 3 — Classify, validate, and flag exemptions

Pueblo's `Zoning` field is blank for most parcels in the open-data endpoint, so
the property category fall-through uses TaxDistrict + LandActualValue heuristics
to distinguish Vacant from Residential/Commercial.


In [ ]:
# Exemption flag
gdf['TaxExempt'] = gdf['TaxExempt'].astype(str).str.strip().str.lower()
gdf['full_exmp'] = gdf['TaxExempt'].isin(['y', 'yes', 'true', '1']).astype(int)

# Property category — limited fields available
def categorize_pueblo(row):
    land = float(row.get('LandActualValue', 0) or 0)
    imp = float(row.get('ImprovementsActualValue', 0) or 0)
    zoning = str(row.get('Zoning', '') or '').strip().upper()
    if row.get('full_exmp', 0) == 1:
        return 'Exempt'
    if imp <= 0:
        return 'Vacant Land'
    if zoning:
        if any(t in zoning for t in ('R-', 'RES', 'SF', 'MF')):
            return 'Residential'
        if any(t in zoning for t in ('COM', 'BUS', 'B-')):
            return 'Commercial'
        if any(t in zoning for t in ('IND', 'I-', 'M-')):
            return 'Industrial'
    # Default: total value heuristic
    total = land + imp
    if total < 1_000_000:
        return 'Residential'
    return 'Commercial'


gdf['PROPERTY_CATEGORY'] = gdf.apply(categorize_pueblo, axis=1)
print(gdf['PROPERTY_CATEGORY'].value_counts())
print(f'\nExempt parcels: {gdf["full_exmp"].sum():,}')

## Section 4 — Current tax (proxy)

Uniform 1.0-mill placeholder — same proxy as the Greeley port. Source's `Tax`
and `LevyURL` fields are blank; the real fix is loading Pueblo County Auditor's
TaxDistrict → consolidated millage table and joining on the `TaxDistrict` code.


In [ ]:
PROXY_MILL_RATE = 1.0
gdf['millage_rate'] = PROXY_MILL_RATE
gdf['taxable_land_value'] = gdf['LandAssessedValue'].clip(lower=0)
gdf['taxable_improvement_value'] = gdf['ImprovementsAssessedValue'].clip(lower=0)
gdf['current_tax'] = (gdf['taxable_land_value'] + gdf['taxable_improvement_value']) * PROXY_MILL_RATE / 1000.0
gdf.loc[gdf['full_exmp'] == 1, 'current_tax'] = 0.0

current_revenue = float(gdf['current_tax'].sum())
print(f'Current revenue (proxy at {PROXY_MILL_RATE} mill): ${current_revenue:,.0f}')

## Section 5 — Split-rate model (2:1)

In [ ]:
solve_df = gdf[gdf['full_exmp'] == 0].copy()
print(f'Solving on {len(solve_df):,} parcels (excluded exempt: {len(gdf) - len(solve_df):,})')

land_millage, improvement_millage, new_revenue, solve_df = model_split_rate_tax(
    df=solve_df,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=float(solve_df['current_tax'].sum()),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

excluded = gdf[gdf['full_exmp'] == 1].copy()
excluded['new_tax'] = 0.0
excluded['tax_change'] = 0.0
excluded['tax_change_pct'] = 0.0
gdf = pd.concat([solve_df, excluded]).sort_index()

print(f'\nLand millage:        {land_millage:.4f}')
print(f'Improvement millage: {improvement_millage:.4f}')
print(f'Revenue check: ${new_revenue:,.0f} vs current ${current_revenue:,.0f}')

In [ ]:
category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title=f'{CITY_NAME} — 2:1 Split-Rate Tax Impact')

## Section 6 — Exploration chart

In [ ]:
import matplotlib
matplotlib.use('Agg')

fig, ax = plt.subplots(figsize=(10, 5))
summary = gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values()
summary.plot.barh(ax=ax, color=np.where(summary < 0, '#228B22', '#8B0000'))
ax.axvline(0, color='black', linewidth=1)
ax.set_title(f'{CITY_NAME} — Median Tax Change % by Category (2:1 Split-Rate)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=120)
plt.close()
print(f'Saved preview chart to {DATA_DIR / "category_preview.png"}')

## Section 7 — Census join + export

In [ ]:
import concurrent.futures

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print('Census API timed out — skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False, census_categories=['Residential'])
print('Done.')